In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Temporal / Multilayer Network Analysis — Italy & US Earthquake Networks

Supplementary notebook extending the main Abe-Suzuki analysis.
Builds one 10-km network per 5-year window (1985–2024) for both catalogs
and tracks how topology, communities, and hubs evolve across windows.

Run as script  : python extras_temporal.py
Convert to notebook: python convert_to_notebook.py extras_temporal.py notebooks/extras_temporal.ipynb
"""

import logging
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.io as pio
import seaborn as sns

from src.temporal import (
    build_temporal_networks,
    compute_temporal_metrics,
    compute_partition_stability,
    compute_hub_persistence,
    compute_edge_turnover,
    plot_temporal_metrics,
    plot_temporal_stability,
    plot_temporal_comparison,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

ITALY_DATA  = Path("data/INGV/italy_earthquakes_1985_2025.csv")
US_DATA     = Path("data/USGS/us_earthquakes_1985_2025.csv")
RESULTS_DIR = Path("results")
CACHE_DIR   = RESULTS_DIR / "cache"

ITALY_CRS = "epsg:32632"
US_CRS    = "epsg:5070"
CELL_SIZE = 10
WINDOW_YEARS = 5
CUT_YEAR  = 1985
TOP_N_HUBS = 20

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "comparison")

## Data Loading

Both catalogs are loaded and filtered to 1985–2024 (complete-decade boundary).
A 5-year window starting at 1985 gives eight non-overlapping windows:
1985–1989, 1990–1994, ..., 2020–2024.

**Why temporal windows?** The static 40-year network aggregates all transitions.
Temporal windows reveal whether the observed scale-free topology, community
structure, and dominant hubs are **stationary** (consistent across decades) or
**episodic** (driven by specific mainshock sequences such as L'Aquila 2009 or
Ridgecrest 2019). A temporally stable network is a stronger scientific claim.

In [ ]:
print("Loading earthquake catalogs...")
df_italy = pd.read_csv(ITALY_DATA)
df_italy["time"] = pd.to_datetime(df_italy["time"], utc=True)
df_italy = (
    df_italy[df_italy["time"].dt.year >= CUT_YEAR]
    .sort_values("time").reset_index(drop=True)
)

df_us = pd.read_csv(US_DATA)
df_us["time"] = pd.to_datetime(df_us["time"], utc=True)
df_us = (
    df_us[df_us["time"].dt.year >= CUT_YEAR]
    .sort_values("time").reset_index(drop=True)
)

print(f"Italy: {len(df_italy):,} events  "
      f"({df_italy['time'].dt.year.min()}–{df_italy['time'].dt.year.max()})")
print(f"US:    {len(df_us):,} events  "
      f"({df_us['time'].dt.year.min()}–{df_us['time'].dt.year.max()})")

## Temporal Network Construction — Italy

Each 5-year window yields an independent Abe-Suzuki network built from that
window's earthquake sequence.  The networks share the same cell grid but
differ in which cells are active and how they are connected — allowing direct
window-to-window comparison of community partitions and hub rankings.

**Runtime note:** each window calls `build_abe_suzuki_network`, which projects
and discretises the catalog (~10–30 s per window depending on event count).
Total runtime for both catalogs: ~5–15 minutes.

In [ ]:
print("\nBuilding Italy temporal networks (5-year windows)...")
italy_temporal = build_temporal_networks(
    df_italy,
    window_years=WINDOW_YEARS,
    cell_size_km=CELL_SIZE,
    target_crs=ITALY_CRS,
    min_events=100,
)
print(f"Italy: {len(italy_temporal)} windows built")
for label, G in italy_temporal:
    print(f"  {label}: {G.number_of_nodes():>5} nodes  {G.number_of_edges():>6} edges")

## Temporal Network Construction — US

In [ ]:
print("\nBuilding US temporal networks (5-year windows)...")
us_temporal = build_temporal_networks(
    df_us,
    window_years=WINDOW_YEARS,
    cell_size_km=CELL_SIZE,
    target_crs=US_CRS,
    min_events=100,
)
print(f"US: {len(us_temporal)} windows built")
for label, G in us_temporal:
    print(f"  {label}: {G.number_of_nodes():>5} nodes  {G.number_of_edges():>6} edges")

## Per-Window Topology Metrics — Italy

Four metrics track topology across windows:
**γ** (power-law exponent) — stable γ across decades supports a stationary
scale-free organisation of seismicity.  Large variation reveals epoch-driven
topology (e.g., a post-mainshock window with anomalously low γ due to hub growth).

**C** (clustering coefficient) — stable C implies the fault network's local
connectivity structure is time-invariant.

**L** (average path length) — stable short L is the small-world signature;
post-mainshock windows may show transient L reduction as new inter-community
edges form.

**⟨k⟩** (mean degree) — tracks overall network density; higher ⟨k⟩ in
windows with major mainshock sequences (increased aftershock activity).

In [ ]:
print("\nComputing Italy per-window topology metrics...")
df_metrics_italy = compute_temporal_metrics(
    italy_temporal, k_min_gamma=5, n_path_samples=100, seed=42
)
display(df_metrics_italy)
plot_temporal_metrics(df_metrics_italy, title="Italy (10 km, 5-year windows)")

## Per-Window Topology Metrics — US

In [ ]:
print("\nComputing US per-window topology metrics...")
df_metrics_us = compute_temporal_metrics(
    us_temporal, k_min_gamma=5, n_path_samples=100, seed=42
)
display(df_metrics_us)
plot_temporal_metrics(df_metrics_us, title="US (10 km, 5-year windows)")

## Italy vs US Metric Comparison

Side-by-side comparison reveals structural differences in temporal dynamics.
Italy's higher γ variance is expected given its episodic mainshock sequences
(L'Aquila 2009, Amatrice 2016) in the 40-year period.  The US network, dominated
by induced seismicity at The Geysers, may show a more stationary γ because
industrial injection rates are more constant than tectonic stress loading.

In [ ]:
for metric, ylabel in [
    ("gamma",           "Power-law exponent γ"),
    ("clustering",      "Clustering coefficient C"),
    ("avg_path_length", "Avg path length L"),
    ("mean_degree",     "Mean degree ⟨k⟩"),
]:
    if metric in df_metrics_italy.columns and metric in df_metrics_us.columns:
        plot_temporal_comparison(
            df_metrics_italy, df_metrics_us,
            metric=metric, ylabel=ylabel,
            title="10 km, 5-year windows",
        )

## Partition Stability — Italy

NMI between Louvain partitions of consecutive windows (restricted to nodes
present in both windows).  NMI = 1 → identical community structure.
NMI ≈ 0 → completely reorganised communities.

High stability implies that tectonic provinces are persistent structural features
of the seismic network.  A stability drop co-occurring with a large mainshock
would indicate that the event reorganised the network's community structure —
testable by checking if the low-NMI window contains a known major earthquake.

In [ ]:
print("\nComputing Italy partition stability across windows...")
df_stability_italy = compute_partition_stability(italy_temporal, seed=42)
display(df_stability_italy)

print("\nComputing US partition stability across windows...")
df_stability_us = compute_partition_stability(us_temporal, seed=42)
display(df_stability_us)

## Hub Persistence — Italy & US

Jaccard similarity of the top-20 PageRank hubs between consecutive windows.
Persistent hubs (high Jaccard) are structurally stable fault segments that
remain topologically dominant across decades — the backbone of seismic activity.
Transient hubs (low Jaccard) appear only in response to specific mainshock
sequences and vanish once the aftershock cluster decays.

In [ ]:
print("\nComputing hub persistence (top-20 PageRank)...")
df_hub_italy = compute_hub_persistence(italy_temporal, top_n=TOP_N_HUBS)
df_hub_us    = compute_hub_persistence(us_temporal,    top_n=TOP_N_HUBS)
display(df_hub_italy.assign(catalog="Italy"))
display(df_hub_us.assign(catalog="US"))

## Edge Turnover — Italy & US

Jaccard similarity of edge sets across consecutive windows.
Low turnover → stable seismic corridors (same cell-to-cell transitions
recur decade after decade, consistent with fixed fault geometry).
High turnover → rapid reorganisation (seismic pathways shift, possibly
driven by stress redistribution after large events).

In [ ]:
print("\nComputing edge turnover...")
df_edge_italy = compute_edge_turnover(italy_temporal)
df_edge_us    = compute_edge_turnover(us_temporal)
display(df_edge_italy.assign(catalog="Italy"))
display(df_edge_us.assign(catalog="US"))

## Stability Summary — Italy

The three stability measures (community NMI, hub Jaccard, edge Jaccard) are
plotted together.  A window where all three drop simultaneously identifies a
seismic reorganisation event — a major mainshock that restructures the network.
Compare the dates of low-NMI / low-Jaccard windows with the catalog of M ≥ 6
events to test whether network reorganisation is mainshock-driven.

In [ ]:
plot_temporal_stability(
    df_stability_italy, df_hub_italy, df_edge_italy,
    title="Italy (10 km, 5-year windows)",
)

## Stability Summary — US

In [ ]:
plot_temporal_stability(
    df_stability_us, df_hub_us, df_edge_us,
    title="US (10 km, 5-year windows)",
)

## Cross-Catalog Stability Comparison

Side-by-side bar chart of mean partition stability for Italy vs US.
A higher mean NMI for Italy would imply that its tectonic communities are
more geographically anchored (stable fault zones), while a higher mean for
the US would suggest that induced-seismicity hubs persist consistently
across decades (industrial injection at fixed geographic locations).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_compare = [
    (df_stability_italy["nmi"],     df_stability_us["nmi"],     "Community NMI",       axes[0]),
    (df_hub_italy["jaccard"],       df_hub_us["jaccard"],       "Hub Jaccard (top-20)", axes[1]),
    (df_edge_italy["jaccard"],      df_edge_us["jaccard"],      "Edge Jaccard",         axes[2]),
]
for italy_vals, us_vals, ylabel, ax in metrics_compare:
    means = [float(np.nanmean(italy_vals)), float(np.nanmean(us_vals))]
    bars = ax.bar(["Italy", "US"], means,
                  color=["#e63946", "#457b9d"], alpha=0.8, edgecolor="k")
    vmax = max(max(means), 1e-4)
    ax.set_ylim(0, vmax * 1.4)
    for bar, v in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, v + vmax * 0.04,
                f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(ylabel, fontsize=11)
    ax.grid(axis="y", ls="--", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Cross-Catalog Stability: Italy vs US (5-year windows)", fontsize=13, y=1.01)
plt.tight_layout()
savefig("temporal_cross_catalog_stability")
plt.show()

## Export Results

All per-window DataFrames are exported to CSV for inclusion in the report.

In [ ]:
out = RESULTS_DIR / "data"
out.mkdir(exist_ok=True)

df_metrics_italy.to_csv(out / "temporal_metrics_italy.csv", index=False)
df_metrics_us.to_csv(out / "temporal_metrics_us.csv", index=False)
df_stability_italy.to_csv(out / "temporal_stability_italy.csv", index=False)
df_stability_us.to_csv(out / "temporal_stability_us.csv", index=False)
df_hub_italy.to_csv(out / "temporal_hub_persistence_italy.csv", index=False)
df_hub_us.to_csv(out / "temporal_hub_persistence_us.csv", index=False)
df_edge_italy.to_csv(out / "temporal_edge_turnover_italy.csv", index=False)
df_edge_us.to_csv(out / "temporal_edge_turnover_us.csv", index=False)
print("\nAll temporal results saved to results/data/")

# Summary table
summary = pd.DataFrame({
    "metric": ["Mean γ", "γ std dev", "Mean C", "Mean L",
                "Mean community NMI", "Mean hub Jaccard", "Mean edge Jaccard"],
    "italy": [
        df_metrics_italy["gamma"].mean(), df_metrics_italy["gamma"].std(),
        df_metrics_italy["clustering"].mean(), df_metrics_italy["avg_path_length"].mean(),
        df_stability_italy["nmi"].mean(), df_hub_italy["jaccard"].mean(),
        df_edge_italy["jaccard"].mean(),
    ],
    "us": [
        df_metrics_us["gamma"].mean(), df_metrics_us["gamma"].std(),
        df_metrics_us["clustering"].mean(), df_metrics_us["avg_path_length"].mean(),
        df_stability_us["nmi"].mean(), df_hub_us["jaccard"].mean(),
        df_edge_us["jaccard"].mean(),
    ],
}).round(4)
print("\n=== Temporal Analysis Summary ===")
display(summary)
summary.to_csv(out / "temporal_summary.csv", index=False)